# Ajuste del NER BERT con MEDDOCAN y evaluación del mecanismo

Sigue la Fase 3 y la Fase 4 de la metodología:
1. Ajusta `dccuchile/bert-base-spanish-wwm-cased` con MEDDOCAN (partición oficial: train para entrenar, dev para validar), con las etiquetas de las que el NER es responsable en el enfoque híbrido (PER, ORG, LOC, ADDRESS).
2. Evalúa las cinco configuraciones (NER, regex, híbrido, híbrido + heurística y Presidio) en los 120 prompts sintéticos de prueba y en MEDDOCAN test.
3. (Opcional) Mide el efecto del enmascaramiento sobre la respuesta con GPT-5.5 Instant y gpt-5-mini.

Antes de empezar: **Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)**.

## 1. Subir el código
Sube `modelo-ner.zip` (la carpeta modelo-ner comprimida) cuando la celda lo pida.

In [ ]:
from google.colab import files
import zipfile, os
subido = files.upload()
nombre = next(iter(subido))
zipfile.ZipFile(nombre).extractall('/content')
os.chdir('/content/modelo-ner')
!ls
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dependencias

In [ ]:
# Cada paquete por separado: si uno falla, los demás igual se instalan
for p in ["presidio-analyzer", "presidio-anonymizer", "accelerate", "sentence-transformers", "openai"]:
    !pip install -q {p}
!python -m spacy download es_core_news_sm -q
import importlib
for m in ["presidio_analyzer", "accelerate", "sentence_transformers", "openai", "transformers", "torch"]:
    importlib.import_module(m)
print("Dependencias listas")

## 3. Ajuste del NER con MEDDOCAN
Tarda unos 15 a 25 minutos en una T4. El reporte de validación queda en `modelos/ner-bert/reporte_validacion.txt`.

In [ ]:
import os
if os.path.exists('modelos/ner-bert/config.json'):
    print('Ya hay un modelo entrenado en este entorno; se omite el entrenamiento.')
else:
    !python scripts/entrenar_ner.py --datos datos/meddocan/train.jsonl --val datos/meddocan/dev.jsonl --base dccuchile/bert-base-spanish-wwm-cased --salida modelos/ner-bert --epocas 4 --batch 16


## 4. Evaluación de la detección (se usa una sola vez sobre los conjuntos de prueba)

In [ ]:
import os
assert os.path.exists("modelos/ner-bert/config.json"), "No hay modelo entrenado: revisa que la celda del paso 3 haya terminado sin errores."
!python scripts/evaluar_deteccion.py --datos datos/sintetico/test.jsonl --nombre sintetico_test_bert --ner-backend hf --ner-modelo modelos/ner-bert
!python scripts/evaluar_deteccion.py --datos datos/meddocan/test.jsonl --nombre meddocan_test_bert --ner-backend hf --ner-modelo modelos/ner-bert

## 5. (Opcional) Efecto sobre la respuesta
Hace 3 llamadas por prompt y por modelo (60 prompts → 360 llamadas). Revisa que el identificador de GPT-5.5 Instant sea el correcto en la documentación de la API antes de correrlo.

In [ ]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
!python scripts/evaluar_respuestas.py --salidas resultados/sintetico_test_bert/salidas_hibrido_heuristica.jsonl --modelos chat-latest gpt-5-mini --n 60

## 6. Descargar resultados y modelo

In [ ]:
!zip -qr resultados_bert.zip resultados modelos/ner-bert/reporte_validacion.txt
files.download('resultados_bert.zip')
# El modelo pesa ~420 MB; para conservarlo, cópialo a Drive:
# from google.colab import drive; drive.mount('/content/drive'); !cp -r modelos/ner-bert /content/drive/MyDrive/